# mTAN-MIMIC: extraccion inicial de cohorte

Este notebook inicia el flujo experimental descrito en el articulo: cada muestra corresponde a una admision hospitalaria de MIMIC-III, con etiqueta de mortalidad intrahospitalaria, contexto tabular a nivel admision/paciente y dos series temporales irregulares dentro de una ventana de 168 horas: frecuencia cardiaca desde `CHARTEVENTS` y glucosa desde `LABEVENTS`.

La primera ejecucion crea una cohorte aleatoria reproducible de 2,000 admisiones. La extraccion completa de eventos temporales queda preparada por chunks porque `CHARTEVENTS.csv` es grande.

In [15]:
from pathlib import Path
import json
import warnings
from typing import Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

MIMIC_DIR = Path("/Volumes/Seagate/MIMIC-III")

# Detecta si el notebook se abrio desde la raiz del proyecto o desde Experimentos.
CWD = Path.cwd()
if CWD.name == "Experimentos":
    EXPERIMENTS_DIR = CWD
elif (CWD / "Experimentos").exists():
    EXPERIMENTS_DIR = CWD / "Experimentos"
else:
    EXPERIMENTS_DIR = Path("Experimentos").resolve()

OUTPUT_DIR = EXPERIMENTS_DIR / "outputs" / "mimic_context_aware_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_VERSION = "2026-05-06-age-overflow-and-subject-split-fix"
RANDOM_SEED = 2026
N_SAMPLES = 2_000
WINDOW_HOURS = 168
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}

HEART_RATE_ITEMIDS = [211, 220045]
GLUCOSE_LABITEMIDS = [50809, 50931]

required_files = {
    "admissions": MIMIC_DIR / "ADMISSIONS.csv",
    "patients": MIMIC_DIR / "PATIENTS.csv",
    "chartevents": MIMIC_DIR / "CHARTEVENTS.csv",
    "labevents": MIMIC_DIR / "LABEVENTS.csv",
    "d_labitems": MIMIC_DIR / "D_LABITEMS.csv",
}
missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"No se encontraron archivos requeridos en {MIMIC_DIR}: {missing}")

print(f"NOTEBOOK_VERSION: {NOTEBOOK_VERSION}")
print(f"MIMIC_DIR: {MIMIC_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
for name, path in required_files.items():
    print(f"{name:12s} {path.stat().st_size / 1024**2:,.1f} MB")

NOTEBOOK_VERSION: 2026-05-06-age-overflow-and-subject-split-fix
MIMIC_DIR: /Volumes/Seagate/MIMIC-III
OUTPUT_DIR: /Users/gabrielamartinez/Library/CloudStorage/GoogleDrive-gabrielam.code@gmail.com/My Drive/1. DCC/Experimentos/outputs/mimic_context_aware_subset
admissions   12.0 MB
patients     2.5 MB
chartevents  33,672.2 MB
labevents    1,768.3 MB
d_labitems   0.0 MB


## 1. Cohorte de admisiones

Se usa `HOSPITAL_EXPIRE_FLAG` como etiqueta binaria y se limita la cohorte base a admisiones con datos en `CHARTEVENTS`. La edad se calcula al momento de admision; edades anonimizadas mayores a 120 se agrupan como 90, siguiendo la convencion habitual para MIMIC-III.

Las particiones `train`, `val` y `test` se asignan por `subject_id`, de modo que todas las admisiones de un mismo paciente dentro de la muestra queden en el mismo grupo.


In [16]:
admission_cols = [
    "SUBJECT_ID", "HADM_ID", "ADMITTIME", "DISCHTIME", "DEATHTIME",
    "ADMISSION_TYPE", "ADMISSION_LOCATION", "INSURANCE", "ETHNICITY",
    "HOSPITAL_EXPIRE_FLAG", "HAS_CHARTEVENTS_DATA",
]
patient_cols = ["SUBJECT_ID", "GENDER", "DOB", "EXPIRE_FLAG"]

admissions = pd.read_csv(
    required_files["admissions"],
    usecols=admission_cols,
    parse_dates=["ADMITTIME", "DISCHTIME", "DEATHTIME"],
)
patients = pd.read_csv(
    required_files["patients"],
    usecols=patient_cols,
    parse_dates=["DOB"],
)


def compute_mimic_age(admit_time: pd.Series, dob: pd.Series) -> pd.Series:
    """Calcula edad sin restar timestamps, evitando overflow por edades anonimizadas."""
    birthday_passed = (
        (admit_time.dt.month > dob.dt.month)
        | (admit_time.dt.month.eq(dob.dt.month) & admit_time.dt.day.ge(dob.dt.day))
    )
    age = (admit_time.dt.year - dob.dt.year - (~birthday_passed).astype(int)).astype(float)
    age = age.mask(age.gt(120), 90)
    return age.clip(lower=0)


def assign_subject_splits(
    sample: pd.DataFrame,
    split_fractions: dict[str, float],
    random_seed: int,
) -> pd.DataFrame:
    """Asigna particiones por subject_id para evitar fuga entre admisiones del mismo paciente."""
    fractions = pd.Series(split_fractions, dtype=float)
    if not np.isclose(fractions.sum(), 1.0):
        raise ValueError(f"Las fracciones de split deben sumar 1.0: {split_fractions}")

    rng = np.random.default_rng(random_seed)
    subject_ids = sample["subject_id"].drop_duplicates().to_numpy()
    rng.shuffle(subject_ids)

    n_subjects = len(subject_ids)
    n_train = int(round(n_subjects * fractions["train"]))
    n_val = int(round(n_subjects * fractions["val"]))

    split_map = {}
    for subject_id in subject_ids[:n_train]:
        split_map[int(subject_id)] = "train"
    for subject_id in subject_ids[n_train:n_train + n_val]:
        split_map[int(subject_id)] = "val"
    for subject_id in subject_ids[n_train + n_val:]:
        split_map[int(subject_id)] = "test"

    sample = sample.copy()
    sample["split"] = sample["subject_id"].map(split_map)
    return sample


cohort = admissions.loc[admissions["HAS_CHARTEVENTS_DATA"].eq(1)].merge(
    patients, on="SUBJECT_ID", how="left", validate="many_to_one"
)
cohort["AGE"] = compute_mimic_age(cohort["ADMITTIME"], cohort["DOB"])
cohort["WINDOW_START"] = cohort["ADMITTIME"]
cohort["WINDOW_END"] = cohort["WINDOW_START"] + pd.to_timedelta(WINDOW_HOURS, unit="h")
cohort = cohort.rename(
    columns={
        "SUBJECT_ID": "subject_id",
        "HADM_ID": "hadm_id",
        "HOSPITAL_EXPIRE_FLAG": "mortality_label",
        "GENDER": "gender",
        "AGE": "age",
        "INSURANCE": "insurance",
        "ADMISSION_TYPE": "admission_type",
        "ADMISSION_LOCATION": "admission_location",
        "ETHNICITY": "ethnicity",
        "ADMITTIME": "admit_time",
        "DISCHTIME": "discharge_time",
        "DEATHTIME": "death_time",
        "WINDOW_START": "window_start",
        "WINDOW_END": "window_end",
    }
)

context_cols = [
    "subject_id", "hadm_id", "admit_time", "discharge_time", "death_time",
    "window_start", "window_end", "mortality_label", "age", "gender",
    "insurance", "admission_type", "admission_location", "ethnicity",
]
cohort = cohort[context_cols].drop_duplicates(subset=["subject_id", "hadm_id"])

if len(cohort) < N_SAMPLES:
    raise ValueError(f"La cohorte base solo tiene {len(cohort)} admisiones; se requieren {N_SAMPLES}.")

sample_admissions = (
    cohort.sample(n=N_SAMPLES, random_state=RANDOM_SEED)
    .sort_values(["subject_id", "hadm_id"])
    .reset_index(drop=True)
)
sample_admissions = assign_subject_splits(sample_admissions, SPLIT_FRACTIONS, RANDOM_SEED)

sample_path = OUTPUT_DIR / "sample_admissions_2000.csv"
sample_admissions.to_csv(sample_path, index=False)

split_summary = (
    sample_admissions.groupby("split")
    .agg(
        admissions=("hadm_id", "size"),
        subjects=("subject_id", "nunique"),
        mortality_rate=("mortality_label", "mean"),
    )
    .reindex(["train", "val", "test"])
)

print(f"Cohorte base: {len(cohort):,} admisiones")
print(f"Muestra guardada: {sample_path}")
print(f"Muestra: {len(sample_admissions):,} admisiones")
print(f"Pacientes unicos en la muestra: {sample_admissions['subject_id'].nunique():,}")
display(sample_admissions.head())
display(sample_admissions["mortality_label"].value_counts(normalize=True).rename("proportion"))
display(split_summary)


Cohorte base: 57,384 admisiones
Muestra guardada: /Users/gabrielamartinez/Library/CloudStorage/GoogleDrive-gabrielam.code@gmail.com/My Drive/1. DCC/Experimentos/outputs/mimic_context_aware_subset/sample_admissions_2000.csv
Muestra: 2,000 admisiones
Pacientes unicos en la muestra: 1,983


,subject_id,hadm_id,admit_time,discharge_time,death_time,window_start,window_end,mortality_label,age,gender,insurance,admission_type,admission_location,ethnicity,split
0,5,178980,2103-02-02 04:31:00,2103-02-04 12:15:00,NaT,2103-02-02 04:31:00,2103-02-09 04:31:00,0,0.0,M,Private,NEWBORN,PHYS REFERRAL/NORMAL DELI,ASIAN,val
1,10,184167,2103-06-28 11:36:00,2103-07-06 12:10:00,NaT,2103-06-28 11:36:00,2103-07-05 11:36:00,0,0.0,F,Medicaid,NEWBORN,PHYS REFERRAL/NORMAL DELI,BLACK/AFRICAN AMERICAN,val
2,18,188822,2167-10-02 11:18:00,2167-10-04 16:15:00,NaT,2167-10-02 11:18:00,2167-10-09 11:18:00,0,50.0,M,Private,EMERGENCY,PHYS REFERRAL/NORMAL DELI,WHITE,train
3,34,115799,2186-07-18 16:46:00,2186-07-20 16:00:00,NaT,2186-07-18 16:46:00,2186-07-25 16:46:00,0,90.0,M,Medicare,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,WHITE,train
4,100,153952,2157-08-10 07:15:00,2157-08-18 19:54:00,NaT,2157-08-10 07:15:00,2157-08-17 07:15:00,0,71.0,F,Private,ELECTIVE,PHYS REFERRAL/NORMAL DELI,UNKNOWN/NOT SPECIFIED,train


mortality_label
0    0.895
1    0.105
Name: proportion, dtype: float64

,admissions,subjects,mortality_rate
split,,,
train,1400,1388,0.110714
val,299,297,0.090301
test,301,298,0.093023


In [17]:
# Pruebas de consistencia de la cohorte inicial.
assert len(sample_admissions) == N_SAMPLES
assert sample_admissions[["subject_id", "hadm_id"]].duplicated().sum() == 0
assert sample_admissions["mortality_label"].isin([0, 1]).all()
assert sample_admissions["window_end"].sub(sample_admissions["window_start"]).dt.total_seconds().eq(WINDOW_HOURS * 3600).all()

# Garantia anti-leakage: un subject_id no puede aparecer en mas de una particion.
subject_split_counts = sample_admissions.groupby("subject_id")["split"].nunique()
assert subject_split_counts.max() == 1
assert sample_admissions["split"].isin(SPLIT_FRACTIONS).all()

summary = {
    "random_seed": RANDOM_SEED,
    "n_samples": int(len(sample_admissions)),
    "n_subjects": int(sample_admissions["subject_id"].nunique()),
    "window_hours": WINDOW_HOURS,
    "mortality_rate": float(sample_admissions["mortality_label"].mean()),
    "age_mean": float(sample_admissions["age"].mean()),
    "age_std": float(sample_admissions["age"].std()),
    "split_counts": split_summary.to_dict(orient="index"),
}
summary_path = OUTPUT_DIR / "sample_admissions_2000_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
summary


{'random_seed': 2026,
 'n_samples': 2000,
 'n_subjects': 1983,
 'window_hours': 168,
 'mortality_rate': 0.105,
 'age_mean': 55.1565,
 'age_std': 27.605761115632852,
 'split_counts': {'train': {'admissions': 1400,
   'subjects': 1388,
   'mortality_rate': 0.11071428571428571},
  'val': {'admissions': 299,
   'subjects': 297,
   'mortality_rate': 0.0903010033444816},
  'test': {'admissions': 301,
   'subjects': 298,
   'mortality_rate': 0.09302325581395349}}}

## 2. Extractor de series temporales irregulares

Cada fila extraida representa una observacion real dentro de la ventana `[0, 168]` horas desde `admit_time`. No se regulariza ni interpola: el timestamp relativo y el valor observado se conservan para que el modelo use mascaras y padding solo en el batching.

In [18]:
def extract_irregular_stream(
    csv_path: Path,
    itemids: list[int],
    variable_name: str,
    source_name: str,
    sample: pd.DataFrame,
    chunksize: int = 1_000_000,
    max_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """Extrae observaciones de una variable para las admisiones muestreadas."""
    usecols = ["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM"]
    target_hadm = set(sample["hadm_id"].astype(int))
    windows = sample[["subject_id", "hadm_id", "window_start", "window_end"]].copy()
    windows["hadm_id"] = windows["hadm_id"].astype(int)

    pieces = []
    rows_seen = 0
    rows_kept = 0

    processed_chunks = 0
    reader = pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize)
    for chunk_idx, chunk in enumerate(reader, start=1):
        if max_chunks is not None and chunk_idx > max_chunks:
            break
        processed_chunks += 1
        rows_seen += len(chunk)

        chunk = chunk.dropna(subset=["HADM_ID", "VALUENUM"]).copy()
        if chunk.empty:
            continue

        chunk["HADM_ID"] = chunk["HADM_ID"].astype(int)
        chunk = chunk.loc[
            chunk["HADM_ID"].isin(target_hadm) & chunk["ITEMID"].isin(itemids)
        ].copy()
        if chunk.empty:
            continue

        chunk["CHARTTIME"] = pd.to_datetime(chunk["CHARTTIME"], errors="coerce")
        chunk = chunk.dropna(subset=["CHARTTIME"])
        chunk = chunk.rename(
            columns={
                "SUBJECT_ID": "subject_id",
                "HADM_ID": "hadm_id",
                "ITEMID": "itemid",
                "CHARTTIME": "chart_time",
                "VALUENUM": "value",
            }
        )
        chunk = chunk.merge(windows, on=["subject_id", "hadm_id"], how="inner")
        chunk = chunk.loc[
            chunk["chart_time"].ge(chunk["window_start"])
            & chunk["chart_time"].le(chunk["window_end"])
        ].copy()
        if chunk.empty:
            continue

        chunk["hours_since_admit"] = (
            chunk["chart_time"] - chunk["window_start"]
        ).dt.total_seconds() / 3600
        chunk["variable"] = variable_name
        chunk["source"] = source_name
        chunk["mask"] = 1
        pieces.append(
            chunk[
                [
                    "subject_id", "hadm_id", "variable", "source", "itemid",
                    "chart_time", "hours_since_admit", "value", "mask",
                ]
            ]
        )
        rows_kept += len(chunk)

    result = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(
        columns=[
            "subject_id", "hadm_id", "variable", "source", "itemid",
            "chart_time", "hours_since_admit", "value", "mask",
        ]
    )
    print(
        f"{variable_name}: chunks={processed_chunks}, "
        f"rows_seen={rows_seen:,}, rows_kept={rows_kept:,}"
    )
    return result.sort_values(["subject_id", "hadm_id", "hours_since_admit"]).reset_index(drop=True)


def stream_coverage(events: pd.DataFrame, sample: pd.DataFrame, variable_name: str) -> pd.Series:
    counts = events.groupby("hadm_id").size().rename(f"{variable_name}_n_obs")
    coverage = sample[["hadm_id"]].merge(counts, on="hadm_id", how="left").fillna(0)
    return coverage[f"{variable_name}_n_obs"].describe()

## 3. Prueba rapida de extraccion

Esta celda prueba el extractor con pocos chunks. Para correr la extraccion completa de las 2,000 admisiones, cambia `RUN_FULL_EXTRACTION = True` en la celda siguiente.

In [19]:
SMOKE_MAX_CHUNKS = 2

hr_smoke = extract_irregular_stream(
    required_files["chartevents"], HEART_RATE_ITEMIDS, "heart_rate", "CHARTEVENTS",
    sample_admissions, chunksize=500_000, max_chunks=SMOKE_MAX_CHUNKS,
)
glucose_smoke = extract_irregular_stream(
    required_files["labevents"], GLUCOSE_LABITEMIDS, "glucose", "LABEVENTS",
    sample_admissions, chunksize=500_000, max_chunks=SMOKE_MAX_CHUNKS,
)

print("heart_rate smoke shape:", hr_smoke.shape)
print("glucose smoke shape:", glucose_smoke.shape)
display(hr_smoke.head())
display(glucose_smoke.head())

assert set(hr_smoke.columns) == set(glucose_smoke.columns)
assert hr_smoke["hours_since_admit"].between(0, WINDOW_HOURS).all() if len(hr_smoke) else True
assert glucose_smoke["hours_since_admit"].between(0, WINDOW_HOURS).all() if len(glucose_smoke) else True

heart_rate: chunks=2, rows_seen=1,000,000, rows_kept=2,101
glucose: chunks=2, rows_seen=1,000,000, rows_kept=542
heart_rate smoke shape: (2101, 9)
glucose smoke shape: (542, 9)


,subject_id,hadm_id,variable,source,itemid,chart_time,hours_since_admit,value,mask
0,1986,160127,heart_rate,CHARTEVENTS,220045,2198-08-26 19:25:00,0.900000,97.0,1
1,1986,160127,heart_rate,CHARTEVENTS,220045,2198-08-26 19:47:00,1.266667,97.0,1
2,1986,160127,heart_rate,CHARTEVENTS,220045,2198-08-26 20:00:00,1.483333,99.0,1
3,1986,160127,heart_rate,CHARTEVENTS,220045,2198-08-26 21:00:00,2.483333,93.0,1
4,1986,160127,heart_rate,CHARTEVENTS,220045,2198-08-27 00:00:00,5.483333,79.0,1


,subject_id,hadm_id,variable,source,itemid,chart_time,hours_since_admit,value,mask
0,18,188822,glucose,LABEVENTS,50931,2167-10-03 05:14:00,17.933333,162.0,1
1,18,188822,glucose,LABEVENTS,50931,2167-10-03 18:30:00,31.200000,316.0,1
2,18,188822,glucose,LABEVENTS,50931,2167-10-04 06:45:00,43.450000,186.0,1
3,34,115799,glucose,LABEVENTS,50931,2186-07-18 18:12:00,1.433333,109.0,1
4,34,115799,glucose,LABEVENTS,50931,2186-07-19 02:56:00,10.166667,164.0,1


In [20]:
RUN_FULL_EXTRACTION = False

if RUN_FULL_EXTRACTION:
    heart_rate = extract_irregular_stream(
        required_files["chartevents"], HEART_RATE_ITEMIDS, "heart_rate", "CHARTEVENTS",
        sample_admissions, chunksize=1_000_000, max_chunks=None,
    )
    glucose = extract_irregular_stream(
        required_files["labevents"], GLUCOSE_LABITEMIDS, "glucose", "LABEVENTS",
        sample_admissions, chunksize=1_000_000, max_chunks=None,
    )
    temporal_events = pd.concat([heart_rate, glucose], ignore_index=True)
    temporal_events = temporal_events.sort_values(
        ["subject_id", "hadm_id", "variable", "hours_since_admit"]
    ).reset_index(drop=True)

    events_path = OUTPUT_DIR / "temporal_events_2000.csv"
    temporal_events.to_csv(events_path, index=False)

    coverage = (
        temporal_events.groupby(["hadm_id", "variable"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    coverage_path = OUTPUT_DIR / "temporal_coverage_2000.csv"
    coverage.to_csv(coverage_path, index=False)

    print(f"Eventos guardados: {events_path}")
    print(f"Cobertura guardada: {coverage_path}")
    display(coverage.describe())
else:
    print("Extraccion completa desactivada. Cambia RUN_FULL_EXTRACTION=True para ejecutarla.")

Extraccion completa desactivada. Cambia RUN_FULL_EXTRACTION=True para ejecutarla.


## 4. Preparacion para P1, P2 y P3

Las fases experimentales comparten la misma representacion temporal irregular. Esta seccion carga los eventos extraidos, normaliza valores usando solo el split de entrenamiento y construye el vector contextual con edad normalizada y variables categoricas codificadas a partir del vocabulario de entrenamiento.

Si todavia no existe `temporal_events_2000.csv`, la seccion usa los eventos de la prueba rapida (`hr_smoke` y `glucose_smoke`) para verificar el pipeline sin ejecutar la extraccion completa de `CHARTEVENTS`.

In [21]:
import math
from dataclasses import dataclass

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        "PyTorch no esta instalado. Ejecuta: python3 -m pip install -r requirements_mtan_mimic.txt"
    ) from exc

try:
    from sklearn.metrics import average_precision_score, roc_auc_score
except ImportError as exc:
    raise ImportError(
        "scikit-learn no esta instalado. Ejecuta: python3 -m pip install -r requirements_mtan_mimic.txt"
    ) from exc

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
VARIABLE_TO_ID = {"heart_rate": 0, "glucose": 1}
ID_TO_VARIABLE = {idx: name for name, idx in VARIABLE_TO_ID.items()}
MAX_SEQ_LEN = 512

print(f"DEVICE: {DEVICE}")

DEVICE: mps


In [22]:
def load_temporal_events_for_model(output_dir: Path) -> pd.DataFrame:
    events_path = output_dir / "temporal_events_2000.csv"
    if events_path.exists():
        events = pd.read_csv(events_path, parse_dates=["chart_time"])
        print(f"Eventos completos cargados: {events_path} ({len(events):,} filas)")
    elif "hr_smoke" in globals() and "glucose_smoke" in globals():
        events = pd.concat([hr_smoke, glucose_smoke], ignore_index=True)
        print(f"Usando eventos smoke para prueba de modelos: {len(events):,} filas")
    else:
        raise FileNotFoundError(
            "No existe temporal_events_2000.csv y tampoco estan disponibles hr_smoke/glucose_smoke. "
            "Ejecuta la prueba rapida o activa RUN_FULL_EXTRACTION=True."
        )

    events = events.loc[events["variable"].isin(VARIABLE_TO_ID)].copy()
    events = events.dropna(subset=["hadm_id", "variable", "hours_since_admit", "value"])
    events["hadm_id"] = events["hadm_id"].astype(int)
    events["subject_id"] = events["subject_id"].astype(int)
    events["variable_id"] = events["variable"].map(VARIABLE_TO_ID).astype(int)
    events = events.sort_values(["hadm_id", "hours_since_admit", "variable_id"]).reset_index(drop=True)
    return events


def normalize_temporal_values(events: pd.DataFrame, admissions_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_hadm = set(admissions_df.loc[admissions_df["split"].eq("train"), "hadm_id"].astype(int))
    train_events = events.loc[events["hadm_id"].isin(train_hadm)]
    stats = (
        train_events.groupby("variable")["value"]
        .agg(mean="mean", std="std")
        .reindex(VARIABLE_TO_ID)
    )
    stats["std"] = stats["std"].replace(0, np.nan).fillna(1.0)
    stats["mean"] = stats["mean"].fillna(events["value"].mean())

    events = events.merge(stats.reset_index(), on="variable", how="left")
    events["value_norm"] = (events["value"] - events["mean"]) / events["std"]
    events["time_norm"] = events["hours_since_admit"] / WINDOW_HOURS
    events["time_norm"] = events["time_norm"].clip(0, 1)
    return events, stats


def build_context_matrix(admissions_df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    context_cols = ["age", "gender", "insurance", "admission_type", "admission_location", "ethnicity"]
    categorical_cols = ["gender", "insurance", "admission_type", "admission_location", "ethnicity"]

    train = admissions_df.loc[admissions_df["split"].eq("train")]
    age_mean = float(train["age"].mean())
    age_std = float(train["age"].std()) or 1.0

    vocab = {}
    for col in categorical_cols:
        values = train[col].fillna("UNKNOWN").astype(str).sort_values().unique().tolist()
        vocab[col] = values + ["__OTHER__"]

    rows = []
    for _, row in admissions_df.iterrows():
        features = [(float(row["age"]) - age_mean) / age_std]
        feature_names = ["age_z"]
        for col in categorical_cols:
            value = "UNKNOWN" if pd.isna(row[col]) else str(row[col])
            categories = vocab[col]
            if value not in categories:
                value = "__OTHER__"
            for category in categories:
                features.append(1.0 if value == category else 0.0)
                feature_names.append(f"{col}={category}")
        rows.append((int(row["hadm_id"]), features))

    context = pd.DataFrame({
        "hadm_id": [r[0] for r in rows],
        "context_features": [np.asarray(r[1], dtype=np.float32) for r in rows],
    })
    metadata = {
        "context_cols": context_cols,
        "categorical_vocab": vocab,
        "age_mean": age_mean,
        "age_std": age_std,
        "feature_names": feature_names,
        "context_dim": len(feature_names),
    }
    return context, metadata


def prepare_model_table(admissions_df: pd.DataFrame, events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    events, temporal_stats = normalize_temporal_values(events, admissions_df)
    context, context_metadata = build_context_matrix(admissions_df)

    available_hadm = set(events["hadm_id"].astype(int))
    model_table = admissions_df.loc[admissions_df["hadm_id"].astype(int).isin(available_hadm)].copy()
    model_table = model_table.merge(context, on="hadm_id", how="left")
    model_table = model_table.sort_values(["split", "subject_id", "hadm_id"]).reset_index(drop=True)

    metadata = {
        "temporal_stats": temporal_stats.reset_index().to_dict(orient="records"),
        "context": context_metadata,
        "n_admissions_with_events": int(len(model_table)),
        "n_events": int(len(events)),
    }
    return model_table, events, metadata


temporal_events_for_model = load_temporal_events_for_model(OUTPUT_DIR)
model_table, temporal_events_for_model, preprocessing_metadata = prepare_model_table(sample_admissions, temporal_events_for_model)

print(f"Admisiones con al menos un evento temporal: {len(model_table):,}")
print(model_table.groupby("split").size())
print(f"Dimension del contexto: {preprocessing_metadata['context']['context_dim']}")
display(model_table.head())

Eventos completos cargados: /Users/gabrielamartinez/Library/CloudStorage/GoogleDrive-gabrielam.code@gmail.com/My Drive/1. DCC/Experimentos/outputs/mimic_context_aware_subset/temporal_events_2000.csv (174,177 filas)
Admisiones con al menos un evento temporal: 1,983
split
test      299
train    1388
val       296
dtype: int64
Dimension del contexto: 52


,subject_id,hadm_id,admit_time,discharge_time,death_time,window_start,window_end,mortality_label,age,gender,insurance,admission_type,admission_location,ethnicity,split,context_features
0,109,110668,2140-08-25 14:39:00,2140-09-02 18:30:00,NaT,2140-08-25 14:39:00,2140-09-01 14:39:00,0,23.0,F,Medicaid,EMERGENCY,EMERGENCY ROOM ADMIT,BLACK/AFRICAN AMERICAN,test,"[-1.1976634, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0..."
1,138,108120,2131-10-31 08:00:00,2131-11-06 12:54:00,NaT,2131-10-31 08:00:00,2131-11-07 08:00:00,0,48.0,M,Private,ELECTIVE,PHYS REFERRAL/NORMAL DELI,WHITE,test,"[-0.27431393, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1...."
2,366,134462,2164-11-18 20:27:00,2164-11-22 15:18:00,NaT,2164-11-18 20:27:00,2164-11-25 20:27:00,0,52.0,M,Medicare,EMERGENCY,EMERGENCY ROOM ADMIT,HISPANIC OR LATINO,test,"[-0.12657802, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0...."
3,395,187218,2128-01-03 15:15:00,2128-01-09 14:59:00,NaT,2128-01-03 15:15:00,2128-01-10 15:15:00,0,74.0,F,Medicare,EMERGENCY,EMERGENCY ROOM ADMIT,WHITE,test,"[0.6859695, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0,..."
4,526,183693,2144-04-01 22:13:00,2144-04-03 18:30:00,NaT,2144-04-01 22:13:00,2144-04-08 22:13:00,0,0.0,F,Private,NEWBORN,PHYS REFERRAL/NORMAL DELI,WHITE,test,"[-2.047145, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,..."


## 5. Dataset y batching irregular

El dataset mantiene las secuencias como listas irregulares de observaciones. El `collate_fn` aplica padding por lote y genera una mascara booleana para que los tokens agregados no contribuyan a la atencion ni al pooling.

In [23]:
class MimicIrregularDataset(Dataset):
    def __init__(self, admissions_df: pd.DataFrame, events_df: pd.DataFrame, max_seq_len: int = 512):
        self.admissions = admissions_df.reset_index(drop=True).copy()
        self.max_seq_len = max_seq_len
        self.events_by_hadm = {
            int(hadm_id): group.sort_values(["hours_since_admit", "variable_id"])
            for hadm_id, group in events_df.groupby("hadm_id")
        }

    def __len__(self) -> int:
        return len(self.admissions)

    def __getitem__(self, idx: int) -> dict:
        row = self.admissions.iloc[idx]
        hadm_id = int(row["hadm_id"])
        events = self.events_by_hadm[hadm_id]
        if len(events) > self.max_seq_len:
            events = events.iloc[: self.max_seq_len]

        return {
            "hadm_id": hadm_id,
            "subject_id": int(row["subject_id"]),
            "values": events["value_norm"].to_numpy(dtype=np.float32),
            "times": events["time_norm"].to_numpy(dtype=np.float32),
            "variable_ids": events["variable_id"].to_numpy(dtype=np.int64),
            "context": row["context_features"].astype(np.float32),
            "label": np.float32(row["mortality_label"]),
        }


def collate_irregular_batch(batch: list[dict]) -> dict:
    batch_size = len(batch)
    lengths = [len(item["values"]) for item in batch]
    max_len = max(lengths)

    values = torch.zeros(batch_size, max_len, dtype=torch.float32)
    times = torch.zeros(batch_size, max_len, dtype=torch.float32)
    variable_ids = torch.zeros(batch_size, max_len, dtype=torch.long)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.float32)
    context = torch.tensor(np.stack([item["context"] for item in batch]), dtype=torch.float32)

    for i, item in enumerate(batch):
        length = lengths[i]
        values[i, :length] = torch.from_numpy(item["values"])
        times[i, :length] = torch.from_numpy(item["times"])
        variable_ids[i, :length] = torch.from_numpy(item["variable_ids"])
        mask[i, :length] = True

    return {
        "values": values,
        "times": times,
        "variable_ids": variable_ids,
        "mask": mask,
        "context": context,
        "labels": labels,
        "hadm_id": [item["hadm_id"] for item in batch],
        "subject_id": [item["subject_id"] for item in batch],
    }


def make_dataloaders(
    admissions_df: pd.DataFrame,
    events_df: pd.DataFrame,
    batch_size: int = 64,
    eval_batch_size: int = 128,
    max_seq_len: int = MAX_SEQ_LEN,
) -> dict[str, DataLoader]:
    loaders = {}
    for split in ["train", "val", "test"]:
        split_admissions = admissions_df.loc[admissions_df["split"].eq(split)].copy()
        if split_admissions.empty:
            continue
        dataset = MimicIrregularDataset(split_admissions, events_df, max_seq_len=max_seq_len)
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size if split == "train" else eval_batch_size,
            shuffle=(split == "train"),
            collate_fn=collate_irregular_batch,
            drop_last=False,
        )
    return loaders

loaders = make_dataloaders(model_table, temporal_events_for_model)
for split, loader in loaders.items():
    batch = next(iter(loader))
    print(split, {k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})

train {'values': (64, 254), 'times': (64, 254), 'variable_ids': (64, 254), 'mask': (64, 254), 'context': (64, 52), 'labels': (64,)}
val {'values': (128, 359), 'times': (128, 359), 'variable_ids': (128, 359), 'mask': (128, 359), 'context': (128, 52), 'labels': (128,)}
test {'values': (128, 244), 'times': (128, 244), 'variable_ids': (128, 244), 'mask': (128, 244), 'context': (128, 52), 'labels': (128,)}


## 6. Modelos P1, P2 y P3

Los tres modelos usan el mismo backbone temporal con proyeccion de valores, embedding de variable, codificacion continua de tiempo, atencion multi-cabeza y pooling enmascarado. La diferencia controlada es:

- `P1TemporalOnly`: solo usa la representacion temporal.
- `P2LateFusion`: concatena la representacion temporal con un encoder MLP de contexto.
- `P3FiLMConditioned`: genera parametros FiLM desde el contexto para modular las capas temporales antes del clasificador.

In [24]:
class ContinuousTimeEncoding(nn.Module):
    def __init__(self, d_model: int, max_period: float = 10_000.0):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model debe ser par para la codificacion sinusoidal")
        frequencies = torch.exp(
            -math.log(max_period) * torch.arange(0, d_model, 2, dtype=torch.float32) / d_model
        )
        self.register_buffer("frequencies", frequencies)

    def forward(self, times: torch.Tensor) -> torch.Tensor:
        angles = times.unsqueeze(-1) * self.frequencies
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)


class TemporalBackbone(nn.Module):
    def __init__(
        self,
        n_variables: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.d_model = d_model
        self.value_projection = nn.Linear(1, d_model)
        self.variable_embedding = nn.Embedding(n_variables, d_model)
        self.time_encoding = ContinuousTimeEncoding(d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_layers)
        ])

    def embed_inputs(self, values: torch.Tensor, times: torch.Tensor, variable_ids: torch.Tensor) -> torch.Tensor:
        value_emb = self.value_projection(values.unsqueeze(-1))
        variable_emb = self.variable_embedding(variable_ids)
        time_emb = self.time_encoding(times)
        return self.dropout(self.input_norm(value_emb + variable_emb + time_emb))

    def encode_layers(self, x: torch.Tensor, mask: torch.Tensor, context: Optional[torch.Tensor] = None) -> torch.Tensor:
        key_padding_mask = ~mask
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=key_padding_mask)
        return x

    def masked_pool(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask_f = mask.unsqueeze(-1).float()
        return (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)

    def forward(self, values: torch.Tensor, times: torch.Tensor, variable_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.embed_inputs(values, times, variable_ids)
        x = self.encode_layers(x, mask)
        return self.masked_pool(x, mask)


class FiLMTemporalBackbone(TemporalBackbone):
    def __init__(self, n_variables: int, context_dim: int, **kwargs):
        super().__init__(n_variables=n_variables, **kwargs)
        self.film_generators = nn.ModuleList([
            nn.Sequential(
                nn.Linear(context_dim, self.d_model),
                nn.GELU(),
                nn.Linear(self.d_model, 2 * self.d_model),
            )
            for _ in self.layers
        ])

    def encode_layers(self, x: torch.Tensor, mask: torch.Tensor, context: Optional[torch.Tensor] = None) -> torch.Tensor:
        if context is None:
            raise ValueError("FiLMTemporalBackbone requiere contexto")
        key_padding_mask = ~mask
        for layer, film_generator in zip(self.layers, self.film_generators):
            x = layer(x, src_key_padding_mask=key_padding_mask)
            gamma_beta = film_generator(context)
            gamma, beta = gamma_beta.chunk(2, dim=-1)
            gamma = 1.0 + torch.tanh(gamma)
            x = gamma.unsqueeze(1) * x + beta.unsqueeze(1)
        return x

    def forward(
        self,
        values: torch.Tensor,
        times: torch.Tensor,
        variable_ids: torch.Tensor,
        mask: torch.Tensor,
        context: torch.Tensor,
    ) -> torch.Tensor:
        x = self.embed_inputs(values, times, variable_ids)
        x = self.encode_layers(x, mask, context=context)
        return self.masked_pool(x, mask)


class ContextEncoder(nn.Module):
    def __init__(self, context_dim: int, hidden_dim: int = 32, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )

    def forward(self, context: torch.Tensor) -> torch.Tensor:
        return self.net(context)


class P1TemporalOnly(nn.Module):
    def __init__(self, n_variables: int, d_model: int = 64, **backbone_kwargs):
        super().__init__()
        self.temporal = TemporalBackbone(n_variables=n_variables, d_model=d_model, **backbone_kwargs)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(backbone_kwargs.get("dropout", 0.2)),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, batch: dict) -> torch.Tensor:
        h_t = self.temporal(batch["values"], batch["times"], batch["variable_ids"], batch["mask"])
        return self.classifier(h_t).squeeze(-1)


class P2LateFusion(nn.Module):
    def __init__(self, n_variables: int, context_dim: int, d_model: int = 64, context_hidden: int = 32, **backbone_kwargs):
        super().__init__()
        self.temporal = TemporalBackbone(n_variables=n_variables, d_model=d_model, **backbone_kwargs)
        self.context_encoder = ContextEncoder(context_dim=context_dim, hidden_dim=context_hidden, dropout=backbone_kwargs.get("dropout", 0.2))
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model + context_hidden),
            nn.Linear(d_model + context_hidden, d_model),
            nn.GELU(),
            nn.Dropout(backbone_kwargs.get("dropout", 0.2)),
            nn.Linear(d_model, 1),
        )

    def forward(self, batch: dict) -> torch.Tensor:
        h_t = self.temporal(batch["values"], batch["times"], batch["variable_ids"], batch["mask"])
        h_c = self.context_encoder(batch["context"])
        return self.classifier(torch.cat([h_t, h_c], dim=-1)).squeeze(-1)


class P3FiLMConditioned(nn.Module):
    def __init__(self, n_variables: int, context_dim: int, d_model: int = 64, **backbone_kwargs):
        super().__init__()
        self.temporal = FiLMTemporalBackbone(
            n_variables=n_variables,
            context_dim=context_dim,
            d_model=d_model,
            **backbone_kwargs,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(backbone_kwargs.get("dropout", 0.2)),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, batch: dict) -> torch.Tensor:
        h_t = self.temporal(
            batch["values"],
            batch["times"],
            batch["variable_ids"],
            batch["mask"],
            batch["context"],
        )
        return self.classifier(h_t).squeeze(-1)

## 7. Entrenamiento y evaluacion

El protocolo usa AdamW, cosine annealing, clipping de gradiente y seleccion por AUROC de validacion. La celda deja `RUN_PHASE_EXPERIMENTS = False` por defecto para evitar entrenar accidentalmente sobre archivos grandes al ejecutar todo el notebook. Cambia esa bandera a `True` cuando ya tengas `temporal_events_2000.csv` completo.

In [25]:
@dataclass
class TrainConfig:
    epochs: int = 20
    batch_size: int = 64
    eval_batch_size: int = 128
    learning_rate: float = 2e-3
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.2


TRAIN_CONFIG = TrainConfig()


def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if torch.is_tensor(value) else value
    return moved


def compute_class_pos_weight(admissions_df: pd.DataFrame) -> torch.Tensor:
    train_labels = admissions_df.loc[admissions_df["split"].eq("train"), "mortality_label"].astype(float)
    positives = train_labels.sum()
    negatives = len(train_labels) - positives
    weight = negatives / max(positives, 1.0)
    return torch.tensor([weight], dtype=torch.float32, device=DEVICE)


def safe_binary_metrics(y_true: np.ndarray, y_score: np.ndarray) -> dict:
    metrics = {"auroc": np.nan, "auprc": np.nan}
    if len(np.unique(y_true)) > 1:
        metrics["auroc"] = float(roc_auc_score(y_true, y_score))
        metrics["auprc"] = float(average_precision_score(y_true, y_score))
    return metrics


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> dict:
    model.eval()
    total_loss = 0.0
    n_samples = 0
    labels_all = []
    scores_all = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, DEVICE)
            logits = model(batch)
            loss = criterion(logits, batch["labels"])
            probs = torch.sigmoid(logits)
            batch_size = len(batch["labels"])
            total_loss += float(loss.item()) * batch_size
            n_samples += batch_size
            labels_all.append(batch["labels"].detach().cpu().numpy())
            scores_all.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(labels_all) if labels_all else np.asarray([])
    y_score = np.concatenate(scores_all) if scores_all else np.asarray([])
    metrics = safe_binary_metrics(y_true, y_score) if len(y_true) else {"auroc": np.nan, "auprc": np.nan}
    metrics["loss"] = total_loss / max(n_samples, 1)
    return metrics


def build_model(phase: str, context_dim: int, config: TrainConfig) -> nn.Module:
    kwargs = {
        "n_variables": len(VARIABLE_TO_ID),
        "d_model": config.d_model,
        "n_heads": config.n_heads,
        "n_layers": config.n_layers,
        "dropout": config.dropout,
    }
    if phase == "P1":
        return P1TemporalOnly(**kwargs)
    if phase == "P2":
        return P2LateFusion(context_dim=context_dim, **kwargs)
    if phase == "P3":
        return P3FiLMConditioned(context_dim=context_dim, **kwargs)
    raise ValueError(f"Fase desconocida: {phase}")


def train_one_phase(
    phase: str,
    admissions_df: pd.DataFrame,
    events_df: pd.DataFrame,
    config: TrainConfig,
) -> tuple[nn.Module, pd.DataFrame, dict]:
    loaders = make_dataloaders(
        admissions_df,
        events_df,
        batch_size=config.batch_size,
        eval_batch_size=config.eval_batch_size,
        max_seq_len=MAX_SEQ_LEN,
    )
    if "train" not in loaders or "val" not in loaders:
        raise ValueError("Se requieren splits train y val con eventos temporales para entrenar.")

    context_dim = int(preprocessing_metadata["context"]["context_dim"])
    model = build_model(phase, context_dim=context_dim, config=config).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=compute_class_pos_weight(admissions_df))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    history = []
    best_state = None
    best_val_auroc = -np.inf
    best_epoch = None

    for epoch in range(1, config.epochs + 1):
        model.train()
        train_loss = 0.0
        n_train = 0
        for batch in loaders["train"]:
            batch = move_batch_to_device(batch, DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch)
            loss = criterion(logits, batch["labels"])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            optimizer.step()
            batch_size = len(batch["labels"])
            train_loss += float(loss.item()) * batch_size
            n_train += batch_size

        scheduler.step()
        val_metrics = evaluate_model(model, loaders["val"], criterion)
        row = {
            "phase": phase,
            "epoch": epoch,
            "train_loss": train_loss / max(n_train, 1),
            "val_loss": val_metrics["loss"],
            "val_auroc": val_metrics["auroc"],
            "val_auprc": val_metrics["auprc"],
            "lr": scheduler.get_last_lr()[0],
        }
        history.append(row)
        print(row)

        score = val_metrics["auroc"]
        if not np.isnan(score) and score > best_val_auroc:
            best_val_auroc = score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_model(model, loaders["test"], criterion) if "test" in loaders else {}
    summary = {
        "phase": phase,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "test_metrics": test_metrics,
    }
    return model, pd.DataFrame(history), summary


def run_phase_experiments(
    admissions_df: pd.DataFrame,
    events_df: pd.DataFrame,
    config: TrainConfig = TRAIN_CONFIG,
    phases: tuple[str, ...] = ("P1", "P2", "P3"),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    histories = []
    summaries = []
    models_dir = OUTPUT_DIR / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    for phase in phases:
        print(f"\n=== Entrenando {phase} ===")
        model, history, summary = train_one_phase(phase, admissions_df, events_df, config)
        histories.append(history)
        summaries.append(summary)
        torch.save(model.state_dict(), models_dir / f"{phase}_best.pt")

    history_df = pd.concat(histories, ignore_index=True)
    summary_df = pd.json_normalize(summaries)
    history_df.to_csv(OUTPUT_DIR / "phase_training_history.csv", index=False)
    summary_df.to_csv(OUTPUT_DIR / "phase_experiment_summary.csv", index=False)
    return history_df, summary_df

In [26]:
# Prueba ligera: verifica que P1, P2 y P3 reciben el mismo batch y producen logits compatibles.
sanity_loader = make_dataloaders(model_table, temporal_events_for_model, batch_size=8, eval_batch_size=8)
sanity_split = "train" if "train" in sanity_loader else next(iter(sanity_loader))
sanity_batch = move_batch_to_device(next(iter(sanity_loader[sanity_split])), DEVICE)
context_dim = int(preprocessing_metadata["context"]["context_dim"])

for phase in ["P1", "P2", "P3"]:
    model = build_model(phase, context_dim=context_dim, config=TRAIN_CONFIG).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(sanity_batch)
    print(phase, logits.shape, torch.sigmoid(logits[:5]).detach().cpu().numpy())
    assert logits.shape == sanity_batch["labels"].shape

P1 torch.Size([8]) [0.5048419 0.5021286 0.5749436 0.5172735 0.5050494]
P2 torch.Size([8]) [0.4845962  0.51703405 0.46793994 0.47179365 0.49502432]
P3 torch.Size([8]) [0.4484796  0.47413367 0.5320715  0.47328013 0.44788086]


In [27]:
RUN_PHASE_EXPERIMENTS = False

if RUN_PHASE_EXPERIMENTS:
    phase_history, phase_summary = run_phase_experiments(
        model_table,
        temporal_events_for_model,
        config=TRAIN_CONFIG,
        phases=("P1", "P2", "P3"),
    )
    display(phase_summary)
else:
    print("Entrenamiento desactivado. Cambia RUN_PHASE_EXPERIMENTS=True para entrenar P1, P2 y P3.")

Entrenamiento desactivado. Cambia RUN_PHASE_EXPERIMENTS=True para entrenar P1, P2 y P3.


## 8. Figura comparativa P1 vs P2 vs P3

Esta figura resume la comparacion central entre las tres fases. La celda lee `phase_experiment_summary.csv` y `phase_training_history.csv`, toma la mejor epoca de validacion de cada fase, y genera una grafica agrupada con AUROC y AUPRC. Si los experimentos se corrieron con eventos `smoke`, la figura es solo una prueba del pipeline; para el articulo debe regenerarse despues de la extraccion y entrenamiento completos.

In [28]:
import os
import sys

MPLCONFIGDIR = OUTPUT_DIR / "matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

if "ipykernel" not in sys.modules:
    os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PHASE_LABELS = {
    "P1": "P1\nTemporal",
    "P2": "P2\nLate fusion",
    "P3": "P3\nFiLM",
}
PHASE_COLORS = {
    "P1": "#4C78A8",
    "P2": "#F58518",
    "P3": "#54A24B",
}


def load_phase_results(output_dir: Path) -> pd.DataFrame:
    summary_path = output_dir / "phase_experiment_summary.csv"
    history_path = output_dir / "phase_training_history.csv"
    if not summary_path.exists() or not history_path.exists():
        raise FileNotFoundError(
            "No se encontraron phase_experiment_summary.csv y phase_training_history.csv. "
            "Primero ejecuta RUN_PHASE_EXPERIMENTS=True."
        )

    summary = pd.read_csv(summary_path)
    history = pd.read_csv(history_path)
    best_rows = []
    for _, row in summary.iterrows():
        phase = row["phase"]
        best_epoch = int(row["best_epoch"])
        hist_row = history.loc[
            history["phase"].eq(phase) & history["epoch"].eq(best_epoch)
        ].iloc[0]
        best_rows.append({
            "phase": phase,
            "best_epoch": best_epoch,
            "validation_auroc": hist_row["val_auroc"],
            "validation_auprc": hist_row["val_auprc"],
            "test_auroc": row.get("test_metrics.auroc", np.nan),
            "test_auprc": row.get("test_metrics.auprc", np.nan),
        })
    results = pd.DataFrame(best_rows)
    phase_order = ["P1", "P2", "P3"]
    results["phase"] = pd.Categorical(results["phase"], categories=phase_order, ordered=True)
    return results.sort_values("phase").reset_index(drop=True)


def plot_phase_comparison(results: pd.DataFrame, figures_dir: Path, prefix: str = "phase_comparison") -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
    panels = [
        ("Validation", "validation_auroc", "validation_auprc"),
        ("Test", "test_auroc", "test_auprc"),
    ]
    x = np.arange(len(results))
    width = 0.34

    for ax, (title, auroc_col, auprc_col) in zip(axes, panels):
        colors = [PHASE_COLORS[str(phase)] for phase in results["phase"]]
        bars_auroc = ax.bar(x - width / 2, results[auroc_col], width, label="AUROC", color=colors, alpha=0.95)
        bars_auprc = ax.bar(x + width / 2, results[auprc_col], width, label="AUPRC", color=colors, alpha=0.45, hatch="//")

        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels([PHASE_LABELS[str(phase)] for phase in results["phase"]], fontsize=9)
        ax.set_ylim(0, 1.0)
        ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.8)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        for bars in [bars_auroc, bars_auprc]:
            for bar in bars:
                height = bar.get_height()
                if np.isnan(height):
                    continue
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    min(height + 0.025, 0.98),
                    f"{height:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    rotation=90 if height > 0.85 else 0,
                )

    axes[0].set_ylabel("Score", fontsize=10)
    axes[1].legend(frameon=False, loc="upper right", fontsize=9)
    fig.suptitle("Comparison of Context Integration Strategies", fontsize=13, fontweight="bold", y=1.03)
    fig.tight_layout()

    png_path = figures_dir / f"{prefix}.png"
    pdf_path = figures_dir / f"{prefix}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    print(f"Figura PNG: {png_path}")
    print(f"Figura PDF: {pdf_path}")
    return png_path

phase_plot_results = load_phase_results(OUTPUT_DIR)
display(phase_plot_results)
phase_comparison_path = plot_phase_comparison(phase_plot_results, FIGURES_DIR)

ModuleNotFoundError: No module named 'matplotlib'